# Demo: the linear-programming nonclassicality solver

This notebook demonstrates the primary solver in `src.solvers.lp_solver`.
It is presentation-only: all logic is imported from `src`.

## Problem

Given a finite set of occupation probabilities, we test whether they are compatible with a classical state, i.e., with a mixture of Poisson distributions,

$$
P_k = \int \mathrm d\mu\, \rho(\mu)\, p_k(\mu),
\qquad
p_k(\mu)=e^{-\mu}\mu^k/k!,
\qquad
\rho(\mu)\ge0.
$$

To this end, we use the witness

$$
f(\boldsymbol\lambda)
=
\max\!\Bigl(0,\max_{\mu\ge0}
\boldsymbol\lambda\cdot\mathbf p(\mu)\Bigr)
-
\boldsymbol\lambda\cdot\mathbf P.
$$

Minimizing $f$ over $\boldsymbol\lambda$ leads to a semi-infinite linear program. The reported quantity

$$
\mathrm{num}=-\min_{\boldsymbol\lambda}f
$$

is positive whenever the supplied probabilities certify nonclassicality in exact arithmetic. In floating-point numerics we should not classify points by testing exactly `num > 0`. The 2D scan routines therefore use a decision threshold `SCAN.bias = 1e-8` by default and mark a point nonclassical only when `num > SCAN.bias`.

For the single-point examples below we use the same threshold explicitly. You can change it locally by setting `decision_threshold = ...` in the notebook, or globally for scan routines by editing `scan.bias` in `config.yaml` before importing `src`. Lower values are more sensitive but less conservative near numerical precision limits; higher values are more conservative.

A probability vector is checked before optimization. If any selected probability is negative, non-finite, or if the selected probabilities already sum to more than one, the point is non-physical rather than nonclassical; the solver returns `num = NaN` and emits a warning.


In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.config import SOLVER, SCAN
from src.solvers.lp_solver import solve_semi_infinite_lp
from src.solvers.minimize_solver import minimize_witness
from src.analytical.jaynes_cummings import Pjaycum
from src.utils.numerics import inverse_factorials

## A single probability vector

We evaluate the witness for one distribution with both solvers.

In [ ]:
indx = np.array([0, 1, 2, 3])
probs = np.array([0.6, 0.1, 0.2, 0.05])
fact = inverse_factorials(int(np.max(indx)))
bounds = [SOLVER.lambda_bound] * len(indx)
decision_threshold = SCAN.bias


t0 = perf_counter()
lp = solve_semi_infinite_lp(probs, indx, fact, bounds)
lp_seconds = perf_counter() - t0

print('decision threshold =', decision_threshold)
print('LP witness value =', round(lp['num'], 8), '-> nonclassical:', lp['num'] > decision_threshold)
print('LP lambda     =', np.round(lp['lam'], 4))
print('LP time [s]   =', round(lp_seconds, 4))

## Nelder-Mead cross-check

The Nelder-Mead solver minimizes the same witness directly. It is useful as an
independent check, but it is a local derivative-free optimizer: the result can
depend on the initial point, and it is typically less efficient than the LP
cutting-plane solver.

In [ ]:
starts = {
    'zero': np.zeros(len(indx)),
    'small': 0.1 * np.ones(len(indx)),
    'ones': np.ones(len(indx)),
    'mixed': np.array([0.3, -0.7, 0.5, -0.2]),
}

nm_rows = []
for name, init_a in starts.items():
    t0 = perf_counter()
    value, lam = minimize_witness(
        probs=probs, indx=indx, fact=fact,
        init_a=init_a,
        init_b=np.ones(len(indx)),
        init_c=0.1 * np.ones(len(indx)),
        bounds=bounds,
    )
    nm_rows.append((name, value, perf_counter() - t0, lam))

print(f"{'start':>8} {'witness':>12} {'time [s]':>10} {'time vs LP [%]':>14} {'LP - NM':>9}")
for name, value, seconds, lam in nm_rows:
    time_delta_pct = 100.0 * (seconds / lp_seconds - 1.0)
    print(f"{name:>8} {value:12.8f} {seconds:10.4f} {time_delta_pct:14.2f} {lp['num'] - value:9.8f}")

best_name, best_value, best_seconds, best_lam = max(nm_rows, key=lambda row: row[1])
print('\nBest Nelder-Mead start:', best_name)
print('Best Nelder-Mead lambda =', np.round(best_lam, 4))
print('LP lambda              =', np.round(lp['lam'], 4))

## Single Jaynes-Cummings point

The same LP witness can be evaluated at one physical-model point. Here we use
the first six photon-number probabilities from the Jaynes-Cummings model and
ask whether that single vector is certified as nonclassical.

In [ ]:
jc_tau = 1.231578947368421
jc_nu = 0.2
jc_pe = 1.0 / 3.0
jc_indx = np.arange(6)

jc_probs = np.array([
    Pjaycum(n=int(n), nu=jc_nu, tau=jc_tau, pe=jc_pe)
    for n in jc_indx
])
jc_fact = inverse_factorials(int(np.max(jc_indx)))
jc_bounds = [SOLVER.lambda_bound] * len(jc_indx)

jc_lp = solve_semi_infinite_lp(
    jc_probs,
    jc_indx,
    jc_fact,
    jc_bounds,
    tol=1e-10,
    max_iter=300,
    imtol=1e-9,
    lp_failed_continue=True,
)

print('Jaynes-Cummings point:')
print('  tau =', jc_tau, '| nu =', jc_nu, '| pe =', round(jc_pe, 4))
print('  indices =', jc_indx.tolist())
print('  probabilities =', np.round(jc_probs, 8))
print('  sum selected probabilities =', round(float(np.sum(jc_probs)), 8))
print('  decision threshold =', decision_threshold)
print('  LP witness value =', round(float(jc_lp['num']), 8), '-> nonclassical:', jc_lp['num'] > decision_threshold)
print('  LP lambda =', np.round(jc_lp['lam'], 4))
print('  solver success =', jc_lp['success'])
if not jc_lp['success']:
    print('  solver message =', jc_lp.get('message'))

## Non-physical input

The supplied occupation probabilities must be physically admissible: all entries must be finite, non-negative, and their sum must not exceed unity,

$$
\sum_i P_i \leq 1.
$$

If these conditions are not satisfied, the probabilities are incompatible with any physical state and the solver will return `NaN` with a warning.

In [ ]:
indx = np.array([0, 1, 2, 3])
probs = np.array([0.6, 0.1, 0.2, 0.3])
fact = inverse_factorials(int(np.max(indx)))
bounds = [SOLVER.lambda_bound] * len(indx)
decision_threshold = SCAN.bias


t0 = perf_counter()
lp = solve_semi_infinite_lp(probs, indx, fact, bounds)
lp_seconds = perf_counter() - t0

print('decision threshold =', decision_threshold)
print('LP witness value =', round(lp['num'], 8), '-> nonclassical:', lp['num'] > decision_threshold)
print('LP lambda     =', np.round(lp['lam'], 4))
print('LP time [s]   =', round(lp_seconds, 4))

**For more complex examples, see the `demo_*` notebooks.**

- `demo_2_analytical_vs_numerical.ipynb`: 2D probability-plane scans, LP versus analytical criteria, and optional `analytical_m`.
- `demo_3_jaynes_cummings.ipynb`: 2D Jaynes-Cummings model scans and boundary plotting for multi-branch regions.
- `demo_4_probabilistic_boson_addition.ipynb`: probabilistic boson addition, reversed y-scan direction, higher-precision LP settings, selected-probability scale diagnostics, and CSV save/load examples.
- `demo_5_counting_matrix_vs_numerics.ipynb`: photon-counting matrix criteria, LP comparison, and analytical M boundary comparison.